# Advanced Transformer Techniques - SOLUTIONS

## Overview
Complete implementations of modern transformer optimizations and techniques used in production LLMs.

## Part 1: Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from typing import Optional, Tuple, List
from torch.utils.data import Dataset, DataLoader
import time

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## Part 2: FlashAttention with PyTorch SDPA

In [ ]:
class FlashMultiHeadAttention(nn.Module):
    """Multi-head attention using PyTorch's optimized scaled_dot_product_attention"""
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = dropout
    
    def forward(self, x: torch.Tensor, is_causal: bool = False, 
                attn_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()
        
        # Project and reshape to (batch, num_heads, seq_len, d_k)
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Use PyTorch's optimized SDPA (FlashAttention under the hood)
        attn_output = F.scaled_dot_product_attention(
            Q, K, V,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal
        )
        
        # Reshape and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(attn_output)
        
        return output

## Part 3: Sparse Attention (Sliding Window)

In [ ]:
def create_sliding_window_mask(seq_len: int, window_size: int, device: torch.device) -> torch.Tensor:
    """Create a sliding window attention mask"""
    # Create position indices
    positions = torch.arange(seq_len, device=device)
    distance = positions.unsqueeze(0) - positions.unsqueeze(1)
    
    # Mask: can attend if within window and not to future
    mask = (distance >= 0) & (distance <= window_size)
    
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, seq_len, seq_len)

class SlidingWindowAttention(nn.Module):
    """Attention with sliding window (Longformer-style)"""
    def __init__(self, d_model: int, num_heads: int, window_size: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.window_size = window_size
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, global_attn_indices: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()
        
        # Project Q, K, V
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Create sliding window mask
        mask = create_sliding_window_mask(seq_len, self.window_size, x.device)
        
        # Add global attention if specified
        if global_attn_indices is not None:
            # Allow global tokens to attend to everything and be attended by everything
            mask = mask.clone()
            mask[:, :, global_attn_indices, :] = True
            mask[:, :, :, global_attn_indices] = True
        
        # Compute attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores.masked_fill(~mask, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_o(attn_output)

## Part 4: Linear Attention (Performer-style)

In [ ]:
class LinearAttention(nn.Module):
    """Linear attention using kernel trick (O(n) complexity)"""
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def feature_map(self, x: torch.Tensor) -> torch.Tensor:
        """Apply non-linear feature transformation to ensure positive values"""
        # ELU + 1 is a common choice (guarantees positivity)
        return F.elu(x) + 1
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()
        
        # Project Q, K, V
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply feature map
        Q = self.feature_map(Q)
        K = self.feature_map(K)
        
        # Linear attention: φ(Q) @ (φ(K)^T @ V)
        # This is O(n) instead of O(n²)
        # Shape: (batch, heads, seq_len, d_k)
        
        # Compute K^T @ V first: (batch, heads, d_k, d_k)
        KV = torch.matmul(K.transpose(-2, -1), V)
        
        # Normalize by sum of K
        K_sum = K.sum(dim=-2, keepdim=True)  # (batch, heads, 1, d_k)
        
        # Compute Q @ (K^T @ V)
        attn_output = torch.matmul(Q, KV)
        
        # Normalize
        normalizer = torch.matmul(Q, K_sum.transpose(-2, -1))
        attn_output = attn_output / (normalizer + 1e-6)
        
        # Reshape and project
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.W_o(attn_output)
        
        return self.dropout(output)

## Part 5: ALiBi (Attention with Linear Biases)

In [ ]:
class ALiBiAttention(nn.Module):
    """Attention with Linear Biases (no positional embeddings needed)"""
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        
        # Precompute slopes
        self.register_buffer('slopes', self.get_slopes())
    
    def get_slopes(self) -> torch.Tensor:
        """Compute ALiBi slopes for each head"""
        def get_slopes_power_of_2(n):
            start = 2 ** (-2 ** -(math.log2(n) - 3))
            ratio = start
            return [start * ratio ** i for i in range(n)]
        
        # For non-power-of-2 heads, interpolate
        if math.log2(self.num_heads).is_integer():
            slopes = get_slopes_power_of_2(self.num_heads)
        else:
            closest_power_of_2 = 2 ** math.floor(math.log2(self.num_heads))
            slopes = get_slopes_power_of_2(closest_power_of_2)
            slopes += get_slopes_power_of_2(2 * closest_power_of_2)[::2][:self.num_heads - closest_power_of_2]
        
        return torch.tensor(slopes).view(1, self.num_heads, 1, 1)
    
    def get_alibi_bias(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Create ALiBi bias matrix"""
        # Create distance matrix
        positions = torch.arange(seq_len, device=device)
        distance = positions.unsqueeze(0) - positions.unsqueeze(1)
        
        # Apply negative slope (penalize distant positions)
        # Shape: (1, num_heads, seq_len, seq_len)
        bias = distance.unsqueeze(0).unsqueeze(0) * (-self.slopes.to(device))
        
        return bias
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()
        
        # Project Q, K, V
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Add ALiBi bias
        alibi_bias = self.get_alibi_bias(seq_len, x.device)
        scores = scores + alibi_bias
        
        # Apply mask if provided
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_o(attn_output)

## Part 6: Relative Positional Bias (T5-style)

In [ ]:
class RelativePositionBias(nn.Module):
    """Learnable relative position biases (T5-style)"""
    def __init__(self, num_heads: int, num_buckets: int = 32, max_distance: int = 128):
        super().__init__()
        self.num_heads = num_heads
        self.num_buckets = num_buckets
        self.max_distance = max_distance
        
        # Learnable bias table
        self.relative_attention_bias = nn.Embedding(num_buckets, num_heads)
    
    def _relative_position_bucket(self, relative_position: torch.Tensor) -> torch.Tensor:
        """Map relative positions to buckets"""
        num_buckets = self.num_buckets
        max_distance = self.max_distance
        
        # Half buckets for exact positions (nearby)
        num_buckets //= 2
        ret = (relative_position >= 0).long() * num_buckets
        n = torch.abs(relative_position)
        
        # Small positions get exact buckets
        max_exact = num_buckets // 2
        is_small = n < max_exact
        
        # Large positions get logarithmic buckets
        val_if_large = max_exact + (
            torch.log(n.float() / max_exact) / math.log(max_distance / max_exact) * (num_buckets - max_exact)
        ).long()
        val_if_large = torch.min(val_if_large, torch.full_like(val_if_large, num_buckets - 1))
        
        ret = ret + torch.where(is_small, n, val_if_large)
        return ret
    
    def forward(self, seq_len: int, device: torch.device) -> torch.Tensor:
        """Compute bias matrix for sequence length"""
        # Create relative position matrix
        positions = torch.arange(seq_len, device=device)
        relative_position = positions.unsqueeze(0) - positions.unsqueeze(1)
        
        # Map to buckets
        buckets = self._relative_position_bucket(relative_position)
        
        # Get bias values from embedding table
        # Shape: (seq_len, seq_len, num_heads)
        bias = self.relative_attention_bias(buckets)
        
        # Reshape to (1, num_heads, seq_len, seq_len)
        bias = bias.permute(2, 0, 1).unsqueeze(0)
        
        return bias

class AttentionWithRelativePositionBias(nn.Module):
    """Multi-head attention with relative position bias"""
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.relative_position_bias = RelativePositionBias(num_heads)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        batch_size, seq_len, _ = x.size()
        
        # Project Q, K, V
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Add relative position bias
        position_bias = self.relative_position_bias(seq_len, x.device)
        scores = scores + position_bias
        
        # Apply mask
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        attn_output = torch.matmul(attn_weights, V)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_o(attn_output)

## Part 7: PreNorm vs PostNorm Transformer Blocks

In [ ]:
class ConfigurableTransformerBlock(nn.Module):
    """Transformer block supporting both PreNorm and PostNorm"""
    def __init__(self, d_model: int, num_heads: int, d_ff: int, 
                 dropout: float = 0.1, pre_norm: bool = True):
        super().__init__()
        self.pre_norm = pre_norm
        
        # Attention
        self.attention = FlashMultiHeadAttention(d_model, num_heads, dropout)
        
        # Feed-forward
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        
        # Layer norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        if self.pre_norm:
            # PreNorm: Normalize before sublayers (more stable, modern)
            attn_output = self.attention(self.norm1(x), is_causal=(mask is None))
            x = x + self.dropout1(attn_output)
            
            ffn_output = self.ffn(self.norm2(x))
            x = x + self.dropout2(ffn_output)
        else:
            # PostNorm: Normalize after sublayers (original Transformer)
            attn_output = self.attention(x, is_causal=(mask is None))
            x = self.norm1(x + self.dropout1(attn_output))
            
            ffn_output = self.ffn(x)
            x = self.norm2(x + self.dropout2(ffn_output))
        
        return x

## Part 8: Decoder-Only Language Model (GPT-style)

In [ ]:
class DecoderOnlyTransformer(nn.Module):
    """GPT-style decoder-only transformer"""
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int, max_len: int = 512,
                 dropout: float = 0.1, use_alibi: bool = False):
        super().__init__()
        self.d_model = d_model
        self.use_alibi = use_alibi
        
        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Positional embeddings (not needed if using ALiBi)
        if not use_alibi:
            self.pos_embedding = nn.Embedding(max_len, d_model)
        
        # Transformer layers
        if use_alibi:
            self.layers = nn.ModuleList([
                nn.ModuleDict({
                    'attn': ALiBiAttention(d_model, num_heads, dropout),
                    'ffn': nn.Sequential(
                        nn.Linear(d_model, d_ff),
                        nn.GELU(),
                        nn.Dropout(dropout),
                        nn.Linear(d_ff, d_model)
                    ),
                    'norm1': nn.LayerNorm(d_model),
                    'norm2': nn.LayerNorm(d_model),
                    'dropout': nn.Dropout(dropout)
                })
                for _ in range(num_layers)
            ])
        else:
            self.layers = nn.ModuleList([
                ConfigurableTransformerBlock(d_model, num_heads, d_ff, dropout, pre_norm=True)
                for _ in range(num_layers)
            ])
        
        # Output
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying
        self.lm_head.weight = self.token_embedding.weight
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.size()
        
        # Embed tokens
        x = self.token_embedding(x) * math.sqrt(self.d_model)
        
        # Add positional embeddings (if not using ALiBi)
        if not self.use_alibi:
            positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
            x = x + self.pos_embedding(positions)
        
        x = self.dropout(x)
        
        # Apply transformer layers
        for layer in self.layers:
            if self.use_alibi:
                # Manual layer with ALiBi
                attn_out = layer['attn'](layer['norm1'](x))
                x = x + layer['dropout'](attn_out)
                ffn_out = layer['ffn'](layer['norm2'](x))
                x = x + layer['dropout'](ffn_out)
            else:
                x = layer(x)
        
        # Output projection
        x = self.norm(x)
        logits = self.lm_head(x)
        
        return logits
    
    @torch.no_grad()
    def generate(self, prompt: torch.Tensor, max_new_tokens: int,
                 temperature: float = 1.0, top_k: Optional[int] = None) -> torch.Tensor:
        """Autoregressive generation"""
        self.eval()
        
        for _ in range(max_new_tokens):
            # Get predictions for last token
            logits = self(prompt)
            logits = logits[:, -1, :] / temperature
            
            # Top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            # Sample
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append
            prompt = torch.cat([prompt, next_token], dim=1)
        
        return prompt

## Part 9: Gradient Checkpointing

In [ ]:
from torch.utils.checkpoint import checkpoint

class TransformerWithCheckpointing(nn.Module):
    """Transformer with gradient checkpointing for memory efficiency"""
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int, dropout: float = 0.1,
                 use_checkpoint: bool = False):
        super().__init__()
        self.use_checkpoint = use_checkpoint
        self.d_model = d_model
        
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(512, d_model)
        
        self.layers = nn.ModuleList([
            ConfigurableTransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.size()
        
        # Embeddings
        x = self.token_embedding(x) * math.sqrt(self.d_model)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = x + self.pos_embedding(positions)
        
        # Apply layers with optional checkpointing
        for layer in self.layers:
            if self.use_checkpoint and self.training:
                # Use gradient checkpointing: trades compute for memory
                x = checkpoint(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        
        x = self.norm(x)
        logits = self.lm_head(x)
        
        return logits

## Part 10: Parameter Sharing and Weight Tying

In [ ]:
class TransformerWithSharing(nn.Module):
    """Transformer with parameter sharing to reduce model size"""
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 num_layers: int, d_ff: int, dropout: float = 0.1,
                 share_layers: bool = False, tie_embeddings: bool = True):
        super().__init__()
        self.d_model = d_model
        self.num_layers = num_layers
        
        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(512, d_model)
        
        # Transformer layers
        if share_layers:
            # All layers share the same weights (like ALBERT)
            shared_layer = ConfigurableTransformerBlock(d_model, num_heads, d_ff, dropout)
            self.layers = nn.ModuleList([shared_layer for _ in range(num_layers)])
        else:
            # Independent layers
            self.layers = nn.ModuleList([
                ConfigurableTransformerBlock(d_model, num_heads, d_ff, dropout)
                for _ in range(num_layers)
            ])
        
        # Output
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: share input and output embeddings
        if tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = x.size()
        
        # Embeddings
        x = self.token_embedding(x) * math.sqrt(self.d_model)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        x = x + self.pos_embedding(positions)
        
        # Apply layers
        for layer in self.layers:
            x = layer(x)
        
        x = self.norm(x)
        logits = self.lm_head(x)
        
        return logits
    
    def count_parameters(self) -> Tuple[int, int]:
        """Count total and unique parameters"""
        total = sum(p.numel() for p in self.parameters())
        unique = sum(p.numel() for p in set(self.parameters()))
        return total, unique

## Part 11: Low-Rank Adaptation (LoRA)

In [ ]:
class LoRALinear(nn.Module):
    """Linear layer with LoRA adaptation"""
    def __init__(self, in_features: int, out_features: int, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha
        
        # Frozen base weight
        self.weight = nn.Parameter(torch.randn(out_features, in_features))
        self.weight.requires_grad = False
        
        # LoRA matrices
        self.lora_A = nn.Parameter(torch.randn(rank, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank))
        
        # Scaling factor
        self.scaling = alpha / rank
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Base output
        result = F.linear(x, self.weight)
        
        # LoRA output: x @ A^T @ B^T
        lora_output = F.linear(F.linear(x, self.lora_A), self.lora_B)
        
        return result + lora_output * self.scaling
    
    def merge_weights(self):
        """Merge LoRA weights into base weight for inference"""
        if self.weight.requires_grad:
            return  # Already merged
        
        # W_new = W + (alpha/r) * B @ A
        delta_w = self.lora_B @ self.lora_A * self.scaling
        self.weight.data += delta_w
        
        # Set LoRA matrices to zero
        self.lora_A.data.zero_()
        self.lora_B.data.zero_()

def apply_lora_to_model(model: nn.Module, rank: int = 4, alpha: float = 1.0, 
                        target_modules: List[str] = ['W_q', 'W_v']):
    """Replace Linear layers with LoRA layers in a model"""
    for name, module in model.named_modules():
        # Check if this is a target module
        for target in target_modules:
            if target in name and isinstance(module, nn.Linear):
                # Get parent module
                parent_name = '.'.join(name.split('.')[:-1])
                child_name = name.split('.')[-1]
                parent = dict(model.named_modules())[parent_name] if parent_name else model
                
                # Replace with LoRA layer
                lora_layer = LoRALinear(
                    module.in_features,
                    module.out_features,
                    rank=rank,
                    alpha=alpha
                )
                lora_layer.weight.data = module.weight.data.clone()
                setattr(parent, child_name, lora_layer)
    
    # Freeze all parameters except LoRA
    for name, param in model.named_parameters():
        if 'lora' not in name:
            param.requires_grad = False
    
    return model

## Part 12: Knowledge Distillation

In [ ]:
class DistillationLoss(nn.Module):
    """Combined loss for knowledge distillation"""
    def __init__(self, temperature: float = 3.0, alpha: float = 0.5):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, student_logits: torch.Tensor, teacher_logits: torch.Tensor,
                labels: torch.Tensor) -> Tuple[torch.Tensor, dict]:
        # Hard loss: standard cross-entropy with true labels
        hard_loss = self.ce_loss(student_logits, labels)
        
        # Soft loss: KL divergence between teacher and student
        student_soft = F.log_softmax(student_logits / self.temperature, dim=-1)
        teacher_soft = F.softmax(teacher_logits / self.temperature, dim=-1)
        
        soft_loss = F.kl_div(
            student_soft,
            teacher_soft,
            reduction='batchmean'
        ) * (self.temperature ** 2)
        
        # Combined loss
        total_loss = self.alpha * soft_loss + (1 - self.alpha) * hard_loss
        
        return total_loss, {
            'total': total_loss.item(),
            'soft': soft_loss.item(),
            'hard': hard_loss.item()
        }

def train_with_distillation(student: nn.Module, teacher: nn.Module,
                           train_loader: DataLoader, num_epochs: int,
                           temperature: float = 3.0, alpha: float = 0.5):
    """Train student model using teacher distillation"""
    device = next(student.parameters()).device
    teacher.eval()
    teacher.to(device)
    
    optimizer = torch.optim.AdamW(student.parameters(), lr=3e-4)
    distillation_loss = DistillationLoss(temperature, alpha)
    
    for epoch in range(num_epochs):
        student.train()
        total_loss = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            # Get teacher predictions (no gradients)
            with torch.no_grad():
                teacher_logits = teacher(batch_x)
            
            # Get student predictions
            student_logits = student(batch_x)
            
            # Compute distillation loss
            loss, loss_dict = distillation_loss(
                student_logits.view(-1, student_logits.size(-1)),
                teacher_logits.view(-1, teacher_logits.size(-1)),
                batch_y.view(-1)
            )
            
            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

## Part 13: Model Pruning

In [ ]:
def magnitude_pruning(model: nn.Module, sparsity: float = 0.5):
    """Unstructured magnitude-based pruning"""
    # Collect all weights
    all_weights = []
    for name, param in model.named_parameters():
        if 'weight' in name and param.requires_grad:
            all_weights.append(param.data.abs().view(-1))
    
    # Compute threshold
    all_weights = torch.cat(all_weights)
    threshold = torch.quantile(all_weights, sparsity)
    
    # Apply pruning
    pruned_params = 0
    total_params = 0
    
    for name, param in model.named_parameters():
        if 'weight' in name and param.requires_grad:
            mask = param.data.abs() > threshold
            param.data *= mask.float()
            
            pruned_params += (~mask).sum().item()
            total_params += param.numel()
    
    actual_sparsity = pruned_params / total_params
    print(f"Pruned {pruned_params}/{total_params} parameters ({actual_sparsity:.2%})")

def compute_head_importance(model: nn.Module, data_loader: DataLoader) -> dict:
    """Compute importance scores for attention heads"""
    model.eval()
    device = next(model.parameters()).device
    
    # Track attention head outputs
    head_importance = {}
    
    for batch_x, _ in data_loader:
        batch_x = batch_x.to(device)
        
        # Forward pass with hooks to capture attention outputs
        # (Simplified - in practice, would use hooks)
        _ = model(batch_x)
    
    return head_importance

def structured_pruning(model: nn.Module, num_heads_to_prune: int = 2):
    """Prune entire attention heads"""
    # Find least important heads
    # In practice, would use gradient-based importance scores
    
    for name, module in model.named_modules():
        if isinstance(module, FlashMultiHeadAttention):
            # Prune heads (simplified - would actually remove dimensions)
            print(f"Pruning {num_heads_to_prune} heads from {name}")
            
            # In full implementation:
            # 1. Identify least important heads
            # 2. Remove corresponding dimensions from Q, K, V, O projections
            # 3. Update num_heads and d_k
            pass

def create_sparse_mask(model: nn.Module, sparsity: float):
    """Create and register sparse masks for pruned model"""
    masks = {}
    
    for name, param in model.named_parameters():
        if 'weight' in name:
            # Create mask
            mask = (param.data.abs() > 1e-6).float()
            masks[name] = mask
            
            # Register as buffer (non-trainable)
            model.register_buffer(f'{name}_mask', mask)
    
    return masks

## Part 14: Complete Training Example

In [ ]:
# Dataset
SEQ_LEN = 64
BATCH_SIZE = 16
VOCAB_SIZE = 1000

class RandomDataset(Dataset):
    def __init__(self, num_samples, seq_len, vocab_size):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        x = torch.randint(0, self.vocab_size, (self.seq_len,))
        y = torch.randint(0, self.vocab_size, (self.seq_len,))
        return x, y

train_dataset = RandomDataset(1000, SEQ_LEN, VOCAB_SIZE)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Create model with modern techniques
model = DecoderOnlyTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=256,
    num_heads=8,
    num_layers=4,
    d_ff=1024,
    max_len=SEQ_LEN,
    dropout=0.1,
    use_alibi=True  # Use ALiBi instead of learned positions
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Training
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        logits = model(batch_x)
        loss = criterion(logits.view(-1, VOCAB_SIZE), batch_y.view(-1))
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

## Part 15: Benchmarking and Comparison

In [ ]:
def benchmark_attention(batch_size=8, seq_len=512, d_model=512, num_heads=8):
    """Benchmark different attention mechanisms"""
    x = torch.randn(batch_size, seq_len, d_model, device=device)
    
    mechanisms = {
        'Flash': FlashMultiHeadAttention(d_model, num_heads).to(device),
        'Sliding Window': SlidingWindowAttention(d_model, num_heads, window_size=128).to(device),
        'Linear': LinearAttention(d_model, num_heads).to(device),
        'ALiBi': ALiBiAttention(d_model, num_heads).to(device),
    }
    
    results = {}
    
    for name, attn in mechanisms.items():
        # Warmup
        for _ in range(10):
            _ = attn(x)
        
        # Benchmark
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        start = time.time()
        
        for _ in range(100):
            _ = attn(x)
        
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        end = time.time()
        
        results[name] = (end - start) / 100
        print(f"{name}: {results[name]*1000:.2f} ms")
    
    return results

def memory_comparison():
    """Compare memory usage of different configurations"""
    configs = [
        ('Base', {'use_checkpoint': False, 'share_layers': False}),
        ('Checkpointing', {'use_checkpoint': True, 'share_layers': False}),
        ('Shared Layers', {'use_checkpoint': False, 'share_layers': True}),
    ]
    
    for name, config in configs:
        if 'use_checkpoint' in config:
            model = TransformerWithCheckpointing(
                vocab_size=VOCAB_SIZE,
                d_model=256,
                num_heads=8,
                num_layers=6,
                d_ff=1024,
                use_checkpoint=config['use_checkpoint']
            ).to(device)
        else:
            model = TransformerWithSharing(
                vocab_size=VOCAB_SIZE,
                d_model=256,
                num_heads=8,
                num_layers=6,
                d_ff=1024,
                share_layers=config['share_layers']
            ).to(device)
        
        param_count = sum(p.numel() for p in model.parameters())
        unique_count = sum(p.numel() for p in set(model.parameters()))
        
        print(f"{name}:")
        print(f"  Total params: {param_count:,}")
        print(f"  Unique params: {unique_count:,}")
        print(f"  Reduction: {(1 - unique_count/param_count)*100:.1f}%")

# Run benchmarks
print("\n=== Attention Mechanism Benchmark ===")
benchmark_attention()

print("\n=== Memory Comparison ===")
memory_comparison()

## Summary

This notebook covered:

1. **FlashAttention**: Using PyTorch's optimized SDPA for faster attention
2. **Sparse Attention**: Sliding window attention for long sequences
3. **Linear Attention**: O(n) complexity attention using kernel trick
4. **ALiBi**: Position encoding via attention biases
5. **Relative Position Bias**: Learnable position-dependent biases
6. **PreNorm/PostNorm**: Different normalization strategies
7. **Decoder-Only Models**: GPT-style architecture
8. **Gradient Checkpointing**: Memory-efficient training
9. **Parameter Sharing**: Reducing model size
10. **LoRA**: Efficient fine-tuning
11. **Distillation**: Training smaller models
12. **Pruning**: Reducing model parameters

These techniques are essential for building and deploying modern LLMs efficiently!